# Bulls & Cows — GRPO Training

Fine-tune **Qwen2.5-1.5B-Instruct** on Bulls-and-Cows puzzles using GRPO (Group Relative Policy Optimization).

Based on the reference [Qwen2.5 3B GRPO notebook](https://github.com/unslothai/unsloth) and adapted for the Bulls & Cows environment.

**Sections:**
1. [Installation](#Installation)
2. [Model Setup](#Model-Setup)
3. [Data Preparation](#Data-Preparation)
4. [Reward Functions](#Reward-Functions)
5. [Training](#Training)
6. [Inference](#Inference)
7. [Evaluation & Bar Chart](#Evaluation)
8. [Save & Push to HuggingFace](#Save)

<a name="Model-Setup"></a>
## 2. Model Setup

Load Qwen2.5-1.5B-Instruct with unsloth 4-bit quantization and LoRA.

In [1]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1,3"
os.environ["CUDA_HOME"] = os.path.expanduser("~/venvs/gpu")
os.environ["CPATH"] = ":".join([
    os.path.expanduser("~/venvs/gpu/include"),
    os.path.expanduser("~/venvs/gpu/targets/x86_64-linux/include"),
])
os.environ["LIBRARY_PATH"] = ":".join([
    os.path.expanduser("~/venvs/gpu/lib"),
    os.path.expanduser("~/venvs/gpu/lib/stubs"),
])

In [2]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
max_prompt_length = 512
max_completion_length = 6000
max_seq_length = max_prompt_length + max_completion_length + 16
lora_rank = 64
# GRPO — 4-bit + SFT LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    #model_name="/home/a.anokhin/buls_and_cow/outputs/sft_warmup/checkpoint-100",  # ← Путь к SFT адаптеру
    model_name="unsloth/Qwen3-4B",
    max_seq_length=max_seq_length,
    load_in_4bit=True,                     # ← 4-bit для GRPO
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.95,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-24 20:34:09 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.15.1.
   \\   /|    NVIDIA H200. Num GPUs = 2. Max memory: 139.811 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. However your setting of `gpu_memory_utilization` will OOM.
Changing `gpu_memory_utilization` to 0.87875.
Unsloth: vLLM loading unsloth/qwen3-4b-unsloth-bnb-4bit with actual GPU utilization = 70.33%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 139.81 GB.
Unsl

/home/a.anokhin/venvs/gpu/lib/python3.11/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


WARNING 02-24 20:34:36 [arg_utils.py:1220] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 02-24 20:34:37 [model.py:541] Resolved architecture: Qwen3ForCausalLM
INFO 02-24 20:34:37 [model.py:1561] Using max model len 6528
INFO 02-24 20:34:37 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'bfloat16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.4.mlp', 'model.layers.6.self_attn', 'model.layers.34.self_attn', 'model.layers.33.self_attn', 'model.layers.4.self_attn', 'model.layers.

/home/a.anokhin/venvs/gpu/lib/python3.11/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 02-24 20:34:41 [topk_topp_sampler.py:47] Using FlashInfer for top-p & top-k sampling.
INFO 02-24 20:34:41 [gpu_model_runner.py:4033] Starting to load model unsloth/qwen3-4b-unsloth-bnb-4bit...
INFO 02-24 20:34:41 [cuda.py:364] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
INFO 02-24 20:34:41 [bitsandbytes_loader.py:786] Loading weights with BitsAndBytes quantization. May take a while ...
INFO 02-24 20:34:43 [weight_utils.py:567] No model.safetensors.index.json found in remote.


...

INFO 02-24 20:35:25 [gpu_model_runner.py:5063] Graph capturing finished in 22 secs, took 2.58 GiB
INFO 02-24 20:35:25 [vllm_utils.py:735] Unsloth: Patched vLLM v1 graph capture finished in 22 secs.


INFO 02-24 20:35:26 [core.py:272] init engine (profile, create kv cache, warmup model) took 41.25 seconds
INFO 02-24 20:35:28 [llm.py:343] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['ffn_norm', 'norm', 'post_layernorm', 'q_norm', 'post_feedforward_layernorm', 'layer_norm2', 'norm2', 'attention_norm', 'layer_norm1', 'pre_feedforward_layernorm', 'k_norm', 'post_attention_layernorm', 'input_layernorm', 'norm1']


Some weights of Qwen3ForCausalLM were not initialized from the model checkpoint at unsloth/qwen3-4b-unsloth-bnb-4bit and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['ffn_norm', 'norm', 'cross_attn_input_layernorm', 'post_layernorm', 'q_norm', 'post_feedforward_layernorm', 'layer_norm2', 'norm2', 'attention_norm', 'layer_norm1', 'pre_feedforward_layernorm', 'k_norm', 'post_attention_layernorm', 'input_layernorm', 'norm1', 'cross_attn_post_attention_layernorm']


Unsloth 2026.2.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

<a name="Data-Preparation"></a>
## 3. Data Preparation

We create a training dataset by:
1. Loading pre-generated JSONL files for difficulties 1–3 (easier puzzles for better learning signal)
2. Optionally generating more on-the-fly via `BullsCowsEnv.generate()`

Each sample is formatted as a chat prompt with system + user messages.

In [4]:
import sys
import json
import random
import re
import os
from datasets import Dataset

sys.path.insert(0, ".")

from bulls_and_cows.bulls_and_cows import BullsCowsEnv
from base.data import Data

SYSTEM_PROMPT = """
Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>
"""

def load_jsonl(path: str) -> list[dict]:
    items = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items


def build_train_dataset(data_dir: str = "data/train",
                        difficulties: list[int] = None) -> Dataset:
    if difficulties is None:
        difficulties = [2, 3, 5, 6]

    all_items = []
    for d in difficulties:
        path = os.path.join(data_dir, f"difficulty_{d}.jsonl")
        if os.path.exists(path):
            items = load_jsonl(path)
            all_items.extend(items[:1000])
            print(f"  Loaded {len(items)} puzzles from {path}")

    print(f"Total training samples: {len(all_items)}")

    random.seed(42)
    random.shuffle(all_items)

    prompts = []
    answers = []
    for item in all_items:
        prompts.append([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["question"]},
        ])
        answers.append(item["answer"])

    return Dataset.from_dict({"prompt": prompts, "answer": answers})


dataset = build_train_dataset()
print(f"\nDataset: {dataset}")
print(f"Sample answer: {dataset[0]['answer']}")


  Loaded 5000 puzzles from data/train/difficulty_2.jsonl
  Loaded 5000 puzzles from data/train/difficulty_3.jsonl
  Loaded 1000 puzzles from data/train/difficulty_5.jsonl
  Loaded 1000 puzzles from data/train/difficulty_6.jsonl
Total training samples: 4000

Dataset: Dataset({
    features: ['prompt', 'answer'],
    num_rows: 4000
})
Sample answer: 8670


<a name="Reward-Functions"></a>
## 4. Reward Functions

We use two types of rewards:
1. **Correctness reward** (2.0) — wraps `BullsCowsEnv.verify()` to check if the extracted answer matches the gold answer   and is consistent with all clues
2. **Format reward** (0.5) — checks that the model uses the `<think>...</think><answer>...</answer>` format

In [ ]:
from bulls_and_cows.scripts.verifier import BullsCowsVerifier

verifier = BullsCowsVerifier()
env = BullsCowsEnv()

# ──────────────────────────────────────────────────────────────
# Answer extraction from <answer>...</answer> tags
# ──────────────────────────────────────────────────────────────
def extract_answer_from_tags(text: str) -> str:
    """Extract answer from <answer>...</answer> tags."""
    if "<answer>" in text and "</answer>" in text:
        answer = text.split("<answer>")[-1].split("</answer>")[0].strip()
        # Try to find \boxed{} inside the answer tags
        boxed = re.findall(r"\\boxed\{([^}]+)\}", answer)
        if boxed:
            return boxed[-1].strip()
        # Otherwise return digits
        digits = re.findall(r"\d+", answer)
        if digits:
            return digits[-1].strip()
        return answer
    # Fallback: use the verifier's extract_answer
    return ""


# ──────────────────────────────────────────────────────────────
# Correctness reward function
# ──────────────────────────────────────────────────────────────
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward 2.0 if the model's answer is correct, 0.0 otherwise."""
    responses = [completion[0]["content"] for completion in completions]
    q = prompts[0][-1]["content"]
    extracted = [extract_answer_from_tags(r) for r in responses]

    rewards = []
    for ext, gold in zip(extracted, answer):
        if ext == gold:
            rewards.append(5.0)
        else:
            rewards.append(0.0)

    # Debug logging (first sample only)
    print(
        "-" * 30,
        f"\nQ: {q[:100]}...",
        f"\nGold: {answer[0]}",
        f"\nExtracted: {extracted[0]}",
        f"\nReward: {rewards[0]}",
    )

    return rewards

def partial_correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Partial reward based on bulls/cows:
    - Each bull (correct digit in correct position) = 0.4
    - Each cow (correct digit in wrong position) = 0.1
    - Max for 3-digit puzzle: 3 bulls = 1.2
    """
    responses = [completion[0]["content"] for completion in completions]
    extracted = [extract_answer_from_tags(r) for r in responses]
    rewards = []
    for ext, gold in zip(extracted, answer):
        # Only score if extracted answer has the right length and is all digits
        if ext.isdigit() and len(ext) == len(gold):
            bulls = sum(e == g for e, g in zip(ext, gold))
            common = sum(min(ext.count(d), gold.count(d)) for d in set(ext))
            cows = common - bulls
            reward = bulls * 0.4 + cows * 0.1
        else:
            reward = 0.0
        rewards.append(reward)
    return rewards


# ──────────────────────────────────────────────────────────────
# Format reward functions
# ──────────────────────────────────────────────────────────────
def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Check for strict <think>\n...\n</think>\n<answer>\n...\n</answer> format."""
    pattern = r"^<think>\n.*?\n</think>\n<answer>\n.*?\n</answer>\n$"
    responses = [c[0]["content"] for c in completions]
    return [0.5 if re.match(pattern, r, re.DOTALL) else 0.0 for r in responses]


def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Check for soft <think>...</think>\s*<answer>...</answer> format."""
    pattern = r"<think>.*?</think>\s*<answer>.*?</answer>"
    responses = [c[0]["content"] for c in completions]
    return [0.5 if re.match(pattern, r, re.DOTALL) else 0.0 for r in responses]


def count_tags(text: str) -> float:
    """Incremental reward for correct XML tag usage."""
    score = 0.0
    if text.count("<think>\n") == 1:
        score += 0.125
    if text.count("\n</think>\n") == 1:
        score += 0.125
    if text.count("\n<answer>\n") == 1:
        score += 0.125
        # Penalize text after </answer>
        score -= len(text.split("\n</answer>\n")[-1]) * 0.001
    if text.count("\n</answer>") == 1:
        score += 0.125
        score -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001
    return score


def tag_count_reward_func(completions, **kwargs) -> list[float]:
    """Reward for partial XML tag correctness."""
    contents = [c[0]["content"] for c in completions]
    return [count_tags(c) for c in contents]


print("Reward functions defined ✅")

Reward functions defined ✅


<a name="Training"></a>
## 5. Training

Configure GRPO and train the model.

In [ ]:
from transformers import TrainerCallback
import pandas as pd
class CsvLoggingCallback(TrainerCallback):
    def __init__(self, save_every=10, path="outputs/training_log.csv"):
        self.save_every = save_every
        self.path = path
    def on_log(self, args, state, control, **kwargs):
        if state.global_step % self.save_every == 0 and state.log_history:
            pd.DataFrame(state.log_history).to_csv(self.path, index=False)
            print(f"[Step {state.global_step}] Saved log to {self.path}")

In [7]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    use_vllm=True,
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.02,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_generations=16,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=4000,
    save_steps=200,
    max_grad_norm=1.00,
    report_to="none",
    output_dir="outputs/rl_after_sft_3",
)

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
    ],
    args=training_args,
    train_dataset=dataset,
    callbacks=[CsvLoggingCallback(save_every=10)],  # ← добавить
)

# ──────────────────────────────────────────────────────────────
# Fix: occasional off-by-one length mismatch between per-token tensors
# (logps/entropy/ratios) and `completion_mask` in Unsloth GRPO.
# This trims/pads per-token tensors to `logits_to_keep` so shapes match.
# ──────────────────────────────────────────────────────────────
import types
import torch
from typing import Optional

if hasattr(trainer, "_get_per_token_logps_and_entropies"):
    _orig_get = trainer._get_per_token_logps_and_entropies

    def _trim_to_keep(x: Optional[torch.Tensor], keep: int) -> Optional[torch.Tensor]:
        if x is None:
            return None
        # Token dimension is dim=1 for (B, T) or (B, T, ...)
        if x.dim() >= 2 and x.size(1) != keep:
            if x.size(1) < keep:
                pad_shape = (x.size(0), keep - x.size(1)) + tuple(x.shape[2:])
                pad = x.new_zeros(pad_shape)
                x = torch.cat([pad, x], dim=1)
            else:
                x = x[:, -keep:]
        return x

    def _patched_get(self, model, input_ids, attention_mask, logits_to_keep, *args, **kwargs):
        logps, entropies = _orig_get(model, input_ids, attention_mask, logits_to_keep, *args, **kwargs)
        logps = _trim_to_keep(logps, logits_to_keep)
        entropies = _trim_to_keep(entropies, logits_to_keep)
        return logps, entropies

    trainer._get_per_token_logps_and_entropies = types.MethodType(_patched_get, trainer)
    print("✅ Patched _get_per_token_logps_and_entropies (shape alignment)")

trainer.train()


✅ Patched _get_per_token_logps_and_entropies (shape alignment)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,000 | Num Epochs = 2 | Total steps = 4,000
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 132,120,576 of 4,154,588,672 (3.18% trained)


WARNING 02-24 20:35:36 [input_processor.py:287] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!
------------------------------ 
Q: Bulls and Cows is a code-breaking game. A secret number of 4 digits has been chosen. Every digit in ... 
Gold: 5932 
Extracted:  
Reward: 0.0


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std,rewards / soft_format_reward_func / mean,rewards / soft_format_reward_func / std,rewards / strict_format_reward_func / mean,rewards / strict_format_reward_func / std
1,0.000000,0.000000,0.000000,6000.000000,6000.000000,6000.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,5649.562500,4119.000000,6000.000000,0.625000,5065.500000,4119.000000,5711.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,6000.000000,6000.000000,6000.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,6000.000000,6000.000000,6000.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,6000.000000,6000.000000,6000.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,6000.000000,6000.000000,6000.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,0.000000,0.000000,0.000000,6000.000000,6000.000000,6000.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,6000.000000,6000.000000,6000.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,5963.593750,4835.000000,6000.000000,0.968750,4835.000000,4835.000000,4835.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
10,0.000000,0.000000,0.000000,6000.000000,6000.000000,6000.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


------------------------------ 
Q: Bulls and Cows is a code-breaking game. A secret number of 4 digits has been chosen. Every digit in ... 
Gold: 5643 
Extracted:  
Reward: 0.0
------------------------------ 
Q: Bulls and Cows is a code-breaking game. A secret number of 4 digits has been chosen. Every digit in ... 
Gold: 9028 
Extracted:  
Reward: 0.0
------------------------------ 
Q: Bulls and Cows is a code-breaking game. A secret number of 5 digits has been chosen. Every digit in ... 
Gold: 87213 
Extracted:  
Reward: 0.0
------------------------------ 
Q: Bulls and Cows is a code-breaking game. A secret number of 5 digits has been chosen. Every digit in ... 
Gold: 57904 
Extracted:  
Reward: 0.0
------------------------------ 
Q: Bulls and Cows is a code-breaking game. A secret number of 3 digits has been chosen. Every digit in ... 
Gold: 941 
Extracted:  
Reward: 0.0
------------------------------ 
Q: Bulls and Cows is a code-breaking game. A secret number of 4 digits has been ch

<a name="Inference"></a>
## 6. Inference

Test the model before and after GRPO training.

In [ ]:
# Save LoRA first
model.save_lora("grpo_saved_lora")
print("LoRA saved to grpo_saved_lora/")

LoRA saved to grpo_saved_lora/


In [ ]:
from vllm import SamplingParams

# Load a sample question from test data
test_items = load_jsonl("data/difficulty_1.jsonl")
sample_q = test_items[0]["question"]
sample_a = test_items[0]["answer"]

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
)

# ── Baseline (no LoRA) ──
text_baseline = tokenizer.apply_chat_template([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": sample_q},
], tokenize=False, add_generation_prompt=True)

output_baseline = model.fast_generate(
    [text_baseline],
    sampling_params=sampling_params,
    lora_request=None,
)[0].outputs[0].text

print("=" * 60)
print("BASELINE (no LoRA)")
print("=" * 60)
print(sample_q)
print("=" * 60)
print(f"Gold answer: {sample_a}")
print(f"Extracted:   {extract_answer_from_tags(output_baseline)}")
print(f"\nResponse:\n{output_baseline}")

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o

BASELINE (no LoRA)
Bulls and Cows is a code-breaking game. A secret number of 3 digits has been chosen. Every digit in the secret number is unique (no repeated digits), and digits are taken from 0-9. The number MAY start with 0.

You are given a series of guesses along with their scores:
- A "bull" means a digit in the guess is correct AND in the correct position.
- A "cow" means a digit in the guess is correct BUT in the wrong position.
- Each digit in the secret and each digit in the guess is used at most once when counting bulls and cows. Bulls are counted first; only the remaining (non-bull) digits can produce cows.

Your task: determine the secret number that is consistent with ALL of the given clues. There is exactly one solution.

Here are the clues:
Guess 1: 219 → 0 bull(s), 2 cow(s)
Guess 2: 872 → 0 bull(s), 0 cow(s)
Guess 3: 684 → 0 bull(s), 1 cow(s)
Guess 4: 926 → 1 bull(s), 1 cow(s)
Guess 5: 108 → 1 bull(s), 0 cow(s)

What is the secret 3-digit number?

Please provide your 

In [ ]:
# ── Trained (with LoRA) ──
output_trained = model.fast_generate(
    [text_baseline],
    sampling_params=sampling_params,
    lora_request=model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

print("=" * 60)
print("TRAINED (with GRPO LoRA)")
print("=" * 60)
print(f"Gold answer: {sample_a}")
print(f"Extracted:   {extract_answer_from_tags(output_trained)}")
print(f"\nResponse:\n{output_trained}")

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o

TRAINED (with GRPO LoRA)
Gold answer: 196
Extracted:   169

Response:
<think>
From Guess 1: 219, we get 2 cows but no bulls. This means 1 and 9 are in the secret but not in the first position.

From Guess 2: 872, we get no bulls or cows. This means none of 8, 7, or 2 are in the secret.

From Guess 3: 684, we get 1 cow. Since 8 and 2 are not in the secret, 6 or 4 must be in the secret but not as the first or second digit (since 1 and 9 are used).

From Guess 4: 926, we get 1 bull and 1 cow. Since 9 cannot be the bull (from Guess 1), 6 must be the bull and 2 the cow. This means 9 is not in the secret, and 2 is in the wrong place, likely the second digit.

From Guess 5: 108, we get 1 bull and 0 cows, confirming 1 is in the secret and in the correct position (first digit).

So the secret number is 169.

</think>
<answer>
169
</answer>
boxed{169}


<a name="Evaluation"></a>
## 7. Evaluation & Bar Chart

Evaluate both the baseline and trained model on each difficulty level (1–10) and plot paired bar charts.

In [ ]:
import numpy as np
from tqdm import tqdm

def evaluate_on_dataset(
    model,
    tokenizer,
    jsonl_path: str,
    lora_request=None,
    max_samples: int = 50,
    batch_size: int = 8,
) -> float:
    """
    Evaluate model accuracy on a JSONL dataset.
    Returns accuracy (0.0 - 1.0).
    """
    items = load_jsonl(jsonl_path)[:max_samples]
    if not items:
        return 0.0

    sampling_params = SamplingParams(
        temperature=0.0,
        top_p=1.0,
        max_tokens=1024,
    )

    correct = 0
    total = 0

    # Process in batches
    for i in range(0, len(items), batch_size):
        batch = items[i:i + batch_size]
        texts = []
        for item in batch:
            text = tokenizer.apply_chat_template([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": item["question"]},
            ], tokenize=False, add_generation_prompt=True)
            texts.append(text)

        outputs = model.fast_generate(
            texts,
            sampling_params=sampling_params,
            lora_request=lora_request,
        )

        for item, output in zip(batch, outputs):
            response = output.outputs[0].text
            extracted = extract_answer_from_tags(response)
            gold = item["answer"]
            if extracted == gold:
                correct += 1
            total += 1

    return correct / total if total > 0 else 0.0


# ── Evaluate on all difficulties ──
difficulties = list(range(1, 2))
baseline_accs = []
trained_accs = []

lora_req = model.load_lora("grpo_saved_lora")

for d in difficulties:
    path = f"data/difficulty_{d}.jsonl"
    print(f"\nEvaluating difficulty {d}...")

    acc_base = evaluate_on_dataset(model, tokenizer, path, lora_request=None)
    acc_train = evaluate_on_dataset(model, tokenizer, path, lora_request=lora_req)

    baseline_accs.append(acc_base)
    trained_accs.append(acc_train)
    print(f"  Baseline: {acc_base:.1%}  |  Trained: {acc_train:.1%}")

# Print summary table
print("\n" + "=" * 50)
print(f"{'Difficulty':>10} {'Baseline':>10} {'Trained':>10} {'Delta':>10}")
print("-" * 50)
for d, b, t in zip(difficulties, baseline_accs, trained_accs):
    print(f"{d:>10} {b:>10.1%} {t:>10.1%} {t-b:>+10.1%}")
print("=" * 50)


Evaluating difficulty 1...


Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%| | 0/11 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(difficulties))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width / 2, baseline_accs, width, label="Qwen2.5-1.5B-Instruct (baseline)", color="#4C72B0")
bars2 = ax.bar(x + width / 2, trained_accs, width, label="Qwen2.5-1.5B-Instruct + GRPO", color="#DD8452")

ax.set_xlabel("Difficulty Level", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Bulls & Cows — Baseline vs GRPO-Trained Model", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([str(d) for d in difficulties])
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f"{height:.0%}",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f"{height:.0%}",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150)
plt.show()
print("Chart saved to accuracy_comparison.png")

ModuleNotFoundError: No module named 'matplotlib'

<a name="Save"></a>
## 8. Save & Push to HuggingFace

Save the trained model and datasets. Uncomment the `push_to_hub` lines and fill in your HF username and token.

In [ ]:
# Save merged model to 16bit
model.save_pretrained_merged(
    "bulls_and_cows_grpo_16bit",
    tokenizer,
    save_method="merged_16bit",
)
print("Model saved to bulls_and_cows_grpo_16bit/")

In [ ]:
# ── Push model to HuggingFace (uncomment and fill in your details) ──
# HF_USERNAME = "your-username"
# HF_TOKEN = "hf_..."
#
# model.push_to_hub_merged(
#     f"{HF_USERNAME}/bulls-and-cows-qwen2.5-1.5b-grpo",
#     tokenizer,
#     save_method="merged_16bit",
#     token=HF_TOKEN,
# )
#
# # Push test datasets
# from datasets import Dataset as HFDataset
# from huggingface_hub import HfApi
#
# api = HfApi(token=HF_TOKEN)
# for d in range(1, 11):
#     api.upload_file(
#         path_or_fileobj=f"data/difficulty_{d}.jsonl",
#         path_in_repo=f"difficulty_{d}.jsonl",
#         repo_id=f"{HF_USERNAME}/bulls-and-cows-test-data",
#         repo_type="dataset",
#     )

print("Done! Uncomment the push_to_hub lines above and fill in your HF credentials.")